# Ylivertainen v2 — Clinical Association Pipeline

End-to-end workflow for finding statistically valid associations between **target outcomes**
and **predictor variables** in a clinical dataset.

This notebook drives six universal modules:

| Module                       | Purpose                                                    |
|------------------------------|------------------------------------------------------------|
| `schema_infer.py`            | Auto-classify each column (continuous, ordinal, …)         |
| `cleaning.py`                | Apply the schema, audit duplicates, derive new columns     |
| `dda.py`                     | Per-column descriptive stats + SVG plots                   |
| `missingness_resolution.py`  | Missing pattern analysis, flags, MICE multiple imputation  |
| `eda.py`                     | Univariate target × predictor screening (FDR-corrected)    |
| `inferential.py`             | Multivariable logistic regression with Rubin pooling       |

**Pipeline order**

```
load → infer schema → clean → DDA → missingness → derive new cols → DDA again
   → EDA screen → MICE impute → multivariable logistic (Rubin pool) → outputs
```

All outputs land under `output/<stage>/{figures,tables}/` as SVG and CSV.


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from schema_infer import (infer_schema, print_schema_template, schema_summary,
                          export_schema_summary, ColSpec)
from cleaning import (apply_schema, audit_duplicates, export_cleaning_artifacts,
                      write_cleaned_csv, bin_numeric, bin_datetime, make_missing_flag,
                      combine_categories)
from dda import run_dda, plot_distribution_by_year
from missingness_resolution import (analyze_missingness, add_missing_flags,
                                    mark_structural_missing, drop_rows,
                                    mice_impute, simple_impute)
from eda import screen_associations
from inferential import run_inferential

OUTPUT_ROOT = Path("output")

# Cohort: set to one calendar year, or None for all years in the file (see §1b).
ANALYSIS_YEARS: list[int] | None = None   # e.g. [2025]

#ANALYSIS_YEARS = [2025]
YEAR_COLUMN = "entry_year"


## 1. Load your data

Change the path to point at your Excel/CSV file. The rest of the notebook is dataset-agnostic.


In [2]:
DATA_PATH = "Meningiomas PSKUS grants - Visi pacienti.csv"   # or "yourdata.csv"

if str(DATA_PATH).endswith(".csv"):
    df_raw = pd.read_csv(DATA_PATH)
else:
    df_raw = pd.read_excel(DATA_PATH)

print(f"Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head(0)

Loaded: 398 rows × 39 columns


,Nr.,Personas kods,Unnamed: 2,"Vecums, gadi","Dzimums, 0 - vīrietis\n1 - sieviete""","Histoloģija, 0 - nav\n1 - ir","WHO pakāpe (2021), 1 / 2 / 3","Progesterons, 0 - negatīvs\n1 - pozitīvs","Ki-67 (%), skaitlis, %","Smadzeņu parenhīmas invāzija, 0 - nav\n1 - ir",...,"Audzēja nekroze, 0 - nav\n1 - ir","Hemorāģiskas sastāvdaļas, 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams","Kaule hiperostoze, 0 - nav\n1 - ir","Kaula invāzija (cortical destruction), 0 - nav\n1 - ir","Tumor Hyperintensity on DWI, 0 - nav\n1 - ir","Tumor Hyperintensity on T2, 0 - nav\n1 - ir","Tumor Hypointensity on T1, 0 - nav\n1 - ir","Sīnuss, 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug","Cauraug falx cerebri 0 - nav, 1 - ir",ADC map value


### 1a. (Optional) Rename columns to clean snake_case

If your source has long Latvian/Russian/free-text column names, rename them here.
Comment this cell out for new datasets where column names are already clean.


In [3]:
# Example for RPE study — edit/remove for other datasets:
# df_raw.columns = ['Gads', 'pk', 'Vecums', 'Pirmsoperācijas_PSA', 'lesion_1_MRI_PIRADS', ...]
# df_raw.columns = [c.strip() for c in df_raw.columns]

#🟧🟧🟧 comfort renaming
COLUMN_RENAME_MAP = {
    "Nr.": "id",
    "Personas kods": "patient_code",
    "Unnamed: 2": "entry_year",

    "Vecums, gadi": "age",
    'Dzimums, 0 - vīrietis\n1 - sieviete"': "sex",
    "Histoloģija, 0 - nav\n1 - ir": "histology_available",
    "WHO pakāpe (2021), 1 / 2 / 3": "who_grade",
    "Progesterons, 0 - negatīvs\n1 - pozitīvs": "progesterone_pos",
    "Ki-67 (%), skaitlis, %": "ki67_pct",

    "Smadzeņu parenhīmas invāzija, 0 - nav\n1 - ir": "brain_invasion",
    "Nekroze histoloģiski, 0 - nav\n1 - ir": "hist_necrosis",

    "MRI izmeklējuma datums": "mri_date",
    "Puse, 1 - labā\n2 - kreisā\n3 - viduslīnija": "side",
    "Lokalizācija: skull base / non–skull base, 0 - non-skull base\n1 - skull base": "tumor_location",
    "Cik meningiomas?": "meningioma_count",

    "Max diametrs, skaitlis,cm": "max_diameter_cm",
    "Tilpums": "tumor_volume",
    "Pamatmodalitāte analīzei, 0 - MRI\n1 - CT\n3 - MRI+CT": "base_modality",
    "K/v i/v, 0 - nav\n1 - ir": "iv_contrast",
    "0 - primārs\n1 - recidīvs": "tumor_episode",

    "Audzēja robeža, 1 = gluda, \n2 = neregulāra": "tumor_margin",
    "Dural tail sign, 0 - nav\n1 - ir": "dural_tail",
    "Gredzenveida kontrastēšanās (tumor capsular enhancement), 0 - nav\n1 - ir": "capsular_enhancement",
    "Kontrastēšanās veids, 0 - homogēna\n1 - heterogēna": "heterogeneous_enhancement",

    "Perifokāla tūska, 0 - nav\n1 - ir": "perifocal_edema",
    "Perifokālas tūskas tilpums, cm3": "edema_volume_cm3",
    "Masas efekts, 0 - nav\n1 - ir": "mass_effect",

    "Audzēja kalcifikācija, 0 - nav\n1 - ir": "calcification",
    "Cistiskas komponentes, 0 - nav\n1 - ir": "cystic_component",
    "Audzēja nekroze, 0 - nav\n1 - ir": "necrosis",
    "Hemorāģiskas sastāvdaļas, 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams ": "hemorrhage",

    "Kaule hiperostoze, 0 - nav\n1 - ir": "hyperostosis",
    "Kaula invāzija (cortical destruction), 0 - nav\n1 - ir": "cortical_destruction",

    "Tumor Hyperintensity on DWI, 0 - nav\n1 - ir": "dwi_hyperintensity",
    "Tumor Hyperintensity on T2, 0 - nav\n1 - ir": "t2_hyperintensity",
    "Tumor Hypointensity on T1, 0 - nav\n1 - ir": "t1_hypointensity",

    "Sīnuss, 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug": "sinus_invasion",
    "Cauraug falx cerebri 0 - nav, 1 - ir": "transfalcine_extension",
    "ADC map value": "adc_value",
    }

df_raw = df_raw.rename(columns=COLUMN_RENAME_MAP)
df = df_raw

#df = df_raw.reindex(['Gads', 'Biopsijas_veids'], axis=1)

df.head(0)


,id,patient_code,entry_year,age,sex,histology_available,who_grade,progesterone_pos,ki67_pct,brain_invasion,...,necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value


### 1b. Cohort filter (optional)

Set `ANALYSIS_YEAR` in **§0 Setup** (e.g. `2025`), then run this cell after column rename.
Re-run the notebook from here through the report if you change the year.

In [4]:
if ANALYSIS_YEARS is not None and len(ANALYSIS_YEARS) == 0:
    raise ValueError("ANALYSIS_YEARS is []; use None for all years or e.g. [2025]")

if ANALYSIS_YEARS is not None:
    n_before = len(df_raw)
    df_raw = df_raw.loc[pd.to_numeric(df_raw[YEAR_COLUMN], errors="coerce").isin(ANALYSIS_YEARS)].copy()
    df = df_raw
    if df_raw.empty:
        raise ValueError(f"No rows with {YEAR_COLUMN} in {ANALYSIS_YEARS!r}")
    print(
        f"Cohort: {YEAR_COLUMN} in {ANALYSIS_YEARS} → "
        f"{len(df_raw)} rows (dropped {n_before - len(df_raw)})"
    )
else:
    years = sorted(pd.to_numeric(df_raw[YEAR_COLUMN], errors="coerce").dropna().astype(int).unique())
    print(f"Cohort: all years {list(years)} — {len(df_raw)} rows")

Cohort: all years [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)] — 398 rows


## 2. Infer schema and override

The engine auto-classifies every column. Inspect the printed dict, copy it into the
next cell, edit anything wrong (e.g. force `lesion_2_MRI_PIRADS` to `ordinal`,
mark `pk` as `id`, drop a junk column with `kind="skip"`).


In [5]:
schema = infer_schema(df_raw)
schema_summary(schema)


,column,kind,keep,ordered_levels,nulls,note
0,id,id,True,None,None,
1,patient_code,id,True,None,None,
2,entry_year,nominal,True,None,None,
3,age,continuous,True,None,None,
4,sex,binary,True,None,None,
5,histology_available,binary,True,None,None,
6,who_grade,nominal,True,None,None,
7,progesterone_pos,binary,True,None,None,
8,ki67_pct,text,True,None,None,
9,brain_invasion,binary,True,None,None,


In [6]:
# Print a paste-back-able template; edit it in the next cell.
print_schema_template(schema);

schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind='id'),
    'entry_year': ColSpec(name='entry_year', kind='nominal'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='binary'),
    'histology_available': ColSpec(name='histology_available', kind='binary'),
    'who_grade': ColSpec(name='who_grade', kind='nominal'),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary'),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    'mri_date': ColSpec(name='mri_date', kind='text'),
    'side': ColSpec(name='side', kind='ordinal', ordered_levels=[1.0, 2.0, 3.0]),
    'tumor_location': ColSpec(name='tumor_location', kind='binary'),
    'meningioma_count': ColSpec(name='meningioma_count', kind='ordinal', ordered_levels=[1.0, 2.0, 3.0, 4.0]

### 2a. Paste the edited schema below

Take the printout from the cell above, paste it here, and adjust kinds/ordered_levels
as needed. Anything you don't override stays as inferred.

For RPE specifically, things to check:
- `pk` → `id`
- `preop_TNM_MDK`, `RPE_TNM` → `nominal`
- `risk_group` → `ordinal` with `ordered_levels=['zema','vidēja','augsta']`
- `biopsy_gleason_grade`, `RPE_grade`, PIRADS columns → `ordinal` with `[1,2,3,4,5]` (or `[0,1,2,3,4,5]`)
- `upgrade`, `upstage`, `downgrade`, `resection_lines_pos` → `binary`


In [7]:
# Example override — adapt to your dataset!
schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind="text", keep=False),
    'entry_year': ColSpec(name='entry_year', kind='datetime', keep=False),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='nominal', replace={0:"male", 1:"female",}),
    'histology_available': ColSpec(name='histology_available', kind='binary', nulls=(2,)),
    'who_grade': ColSpec(name='who_grade', kind='ordinal', ordered_levels=["1","2","3"]),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary', nulls=(2,)),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    'mri_date': ColSpec(name='mri_date', kind='datetime', keep=False),
    'side': ColSpec(name='side', kind='nominal', replace={1: "right", 2: "left", 3: "midline"}),
    'tumor_location': ColSpec(name='tumor_location', kind='nominal', replace={0: "non_skull_base", 1: "skull_base"}, nulls=(2,)),
    'meningioma_count': ColSpec(name='meningioma_count', kind='ordinal', ordered_levels=[1,2,3,4,5]),
    'max_diameter_cm': ColSpec(name='max_diameter_cm', kind='continuous'),
    'tumor_volume': ColSpec(name='tumor_volume', kind='continuous'),
    'base_modality': ColSpec(name='base_modality', kind='nominal', replace={0: "mri", 1: "ct", 3: "mri_ct"}),
    'iv_contrast': ColSpec(name='iv_contrast', kind='binary'),
    'tumor_episode': ColSpec(name='tumor_episode', kind='nominal', replace={0: "primary", 1: "recurrent"}, nulls=(2, "multiplas")),
    'tumor_margin': ColSpec(name='tumor_margin', kind='nominal', replace={1: "regular", 2: "irregular"}, nulls=(0,)),
    'dural_tail': ColSpec(name='dural_tail', kind='binary'),
    'capsular_enhancement': ColSpec(name='capsular_enhancement', kind='binary'),
    'heterogeneous_enhancement': ColSpec(name='heterogeneous_enhancement', kind='binary'),
    'perifocal_edema': ColSpec(name='perifocal_edema', kind='binary'),
    'edema_volume_cm3': ColSpec(name='edema_volume_cm3', kind='continuous'),
    'mass_effect': ColSpec(name='mass_effect', kind='binary'),
    'calcification': ColSpec(name='calcification', kind='binary'),
    'cystic_component': ColSpec(name='cystic_component', kind='binary'),
    'necrosis': ColSpec(name='necrosis', kind='binary'),
    'hemorrhage': ColSpec(name='hemorrhage', kind='binary', nulls=(2.0,)),
    'hyperostosis': ColSpec(name='hyperostosis', kind='binary'),
    'cortical_destruction': ColSpec(name='cortical_destruction', kind='binary'),
    'dwi_hyperintensity': ColSpec(name='dwi_hyperintensity', kind='binary'),
    't2_hyperintensity': ColSpec(name='t2_hyperintensity', kind='binary'),
    't1_hypointensity': ColSpec(name='t1_hypointensity', kind='binary'),
    'sinus_invasion': ColSpec(name='sinus_invasion', kind='ordinal', replace={0: "no_invasion", 1: "sinus_invasion", 2: "transsinus_extension"}, ordered_levels=["no_invasion", "sinus_invasion", "transsinus_extension"]),
    'transfalcine_extension': ColSpec(name='transfalcine_extension', kind='binary'),
    'adc_value': ColSpec(name='adc_value', kind='continuous'),
    }
# merge overrides on top of inferred schema:
schema.update(schema_overrides)

schema_summary(schema)
export_schema_summary(schema, OUTPUT_ROOT)


PosixPath('output/schema/schema_summary.csv')

## 3. Apply schema → coerce dtypes, replacements, nulls

This is the only place dtypes are set. Downstream stages trust the schema.


In [8]:
schema_log = []
df = apply_schema(df_raw, schema, log=schema_log)
n_rows_after_schema = len(df)
df.dtypes
df.isna().sum().reset_index()

,index,0
0,id,0
1,patient_code,0
2,entry_year,5
3,age,1
4,sex,1
5,histology_available,1
6,who_grade,32
7,progesterone_pos,33
8,ki67_pct,32
9,brain_invasion,32


## 4. Duplicate audit

Provide ID columns. The audit returns rows in duplicate groups and a cleaned frame.


In [9]:
ID_COLS = ['id', 'patient_code', 'entry_year']   # edit for your dataset

dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
print(f"Found {len(dupes)} duplicated rows (across {len(dupes)//2 if len(dupes) else 0}+ groups)")
dupes.head()


Found 0 duplicated rows (across 0+ groups)


,id,patient_code,entry_year,age,sex,histology_available,who_grade,progesterone_pos,ki67_pct,brain_invasion,...,necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value


## 4b. Row removal — drop invalid records

After dtypes are coerced and duplicates audited, this is the place to permanently
remove rows that should not be in analysis at all (data-entry errors, ineligible
patients, impossible values). Every drop is **logged** so the methods section of
your paper can quote exact counts.

Common reasons:
- Out-of-range values (e.g. `vecums < 18`, `preop_PSA < 0`, `biopsy_to_RPE_days < 0`)
- Wrong cohort (e.g. patients without a primary RPE)
- Records missing critical identifiers
- Failed sanity checks against source records

Use `where=` for readable pandas-query strings, or `mask=` for arbitrary boolean
Series. Add as many calls as you need.


In [10]:
# ─────────────────────────────────────────────────────────────────────────
# Row removal — delete rows that should not be in the analysis at all.
#
# This is for rows that are STRUCTURALLY WRONG (out-of-cohort patient, data-
# entry errors, impossible values), NOT for rows with missing values
# (missingness is handled in section 6). Every call appends an entry to
# drop_log so you can report exact counts in your paper's methods section.
#
# Two ways to specify which rows to drop:
#   where='vecums < 18'                    → pandas query string (most readable)
#   mask=df['Gleason_grade_pēc_RPE'].isna() & ...   → arbitrary boolean Series (most powerful)
#
# Always provide a meaningful `reason` — it shows up in the audit log.
# ─────────────────────────────────────────────────────────────────────────

drop_log = []

df = drop_rows(df, mask=df['who_grade'].isna(),
                reason='who_grade is NaN', log=drop_log)

# --- summary table ---
pd.DataFrame(drop_log) if drop_log else print('No rows dropped (uncomment examples above as needed)')

export_cleaning_artifacts(
    OUTPUT_ROOT,
    df=df,
    n_rows_raw=len(df_raw),
    n_rows_after_schema=n_rows_after_schema,
    n_rows_final=len(df),
    schema=schema,
    drop_log=drop_log,
    dupes=dupes,
    schema_log=schema_log,
    )


{'summary': PosixPath('output/cleaning/cleaning_summary.csv'),
 'cleaned': PosixPath('output/cleaning/cleaned.csv'),
 'log': PosixPath('output/cleaning/cleaning_log.csv')}

## 5. DDA — first pass

Descriptive stats + SVG plots for every kept column.
Outputs land in `output/dda/`.


In [11]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

print("\n--- continuous ---");  display(dda_tables['continuous'])
print("\n--- categorical ---"); display(dda_tables['categorical'])
print("\n--- binary ---");      display(dda_tables['binary'])
print("\n--- datetime ---");    display(dda_tables['datetime'])



--- continuous ---


,column,kind,n,n_unique,missing_pct,min,p_5th,median,mean,trimmed_mean,p_95th,max,mode,std,cv,iqr,skewness,kurtosis
0,age,continuous,366,61,0.00,20.000,40.0000,64.50,63.021858,63.636054,81.0000,92.0,NaN,12.755641,0.202400,17.7500,-0.457413,-0.072710
1,max_diameter_cm,continuous,262,85,28.42,1.210,1.8000,3.80,4.135076,4.003810,7.4000,9.2,3.00,1.756800,0.424853,2.5000,0.602967,-0.423890
2,tumor_volume,continuous,286,251,21.86,0.189,1.7075,14.40,28.686105,22.407435,100.6500,168.0,NaN,33.117038,1.154463,32.7875,1.602898,1.947349
3,edema_volume_cm3,continuous,22,20,93.99,0.000,0.0000,17.40,30.758182,24.426667,100.7600,135.0,0.00,37.202721,1.209523,42.8750,1.375768,1.108988
4,adc_value,continuous,236,69,35.52,0.410,0.6150,0.83,0.844763,0.832084,1.1625,1.7,0.79,0.163424,0.193456,0.1700,1.072887,3.349694



--- categorical ---


,column,kind,ordered,n,n_unique,missing_pct,first_mode,first_mode_pct,second_mode,second_mode_pct,rarest,max_class_imbalance,median_category,balance,entropy_bin
0,sex,nominal,False,366,2,0.00,female,69.13,male,30.87,male,2.24,NaN,0.8917,0.8917
1,who_grade,ordinal,True,366,3,0.00,1,69.95,2,27.05,3,23.27,1,0.6454,1.0229
2,side,nominal,False,267,3,27.05,left,47.94,right,44.19,midline,6.10,NaN,0.8314,1.3177
3,tumor_location,nominal,False,262,2,28.42,non_skull_base,53.82,skull_base,46.18,skull_base,1.17,NaN,0.9958,0.9958
4,meningioma_count,ordinal,True,300,4,18.03,1,90.33,2,7.33,5,NaN,1,0.2778,0.5556
5,base_modality,nominal,False,266,3,27.32,mri,73.31,mri_ct,25.56,ct,65.00,NaN,0.5706,0.9044
6,tumor_episode,nominal,False,237,2,35.25,0,90.72,1,9.28,1,9.77,NaN,0.4458,0.4458
7,tumor_margin,nominal,False,263,2,28.14,regular,59.32,irregular,40.68,irregular,1.46,NaN,0.9748,0.9748
8,sinus_invasion,ordinal,True,244,3,33.33,no_invasion,73.77,sinus_invasion,18.44,transsinus_extension,9.47,no_invasion,0.6690,1.0603



--- binary ---


,column,kind,ordered,n,n_unique,missing_pct,first_mode,first_mode_pct,second_mode,second_mode_pct,rarest,max_class_imbalance,median_category,balance,entropy_bin
0,histology_available,binary,False,366,2,0.00,True,99.73,False,0.27,False,365.00,NaN,0.0272,0.0272
1,progesterone_pos,binary,False,365,2,0.27,True,97.53,False,2.47,False,39.56,NaN,0.1668,0.1668
2,brain_invasion,binary,False,366,2,0.00,False,98.36,True,1.64,True,60.00,NaN,0.1207,0.1207
3,hist_necrosis,binary,False,366,2,0.00,False,90.44,True,9.56,True,9.46,NaN,0.4550,0.4550
4,iv_contrast,binary,False,271,2,25.96,True,97.42,False,2.58,False,37.71,NaN,0.1730,0.1730
5,dural_tail,binary,False,260,2,28.96,True,80.77,False,19.23,False,4.20,NaN,0.7063,0.7063
6,capsular_enhancement,binary,False,259,2,29.23,True,85.71,False,14.29,False,6.00,NaN,0.5917,0.5917
7,heterogeneous_enhancement,binary,False,258,2,29.51,True,58.91,False,41.09,False,1.43,NaN,0.9769,0.9769
8,perifocal_edema,binary,False,261,2,28.69,True,77.01,False,22.99,False,3.35,NaN,0.7778,0.7778
9,mass_effect,binary,False,260,2,28.96,True,85.38,False,14.62,False,5.84,NaN,0.6001,0.6001



--- datetime ---


""


## 6. Missingness analysis

Per-column %, plus a Jaccard co-missingness heatmap so you can spot blocks of
columns that are missing together (often a data-entry-process artifact).


In [12]:
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary

,column,n_missing,pct_missing
0,transfalcine_extension,353,96.45
1,edema_volume_cm3,344,93.99
2,adc_value,130,35.52
3,tumor_episode,129,35.25
4,sinus_invasion,122,33.33
5,t2_hyperintensity,117,31.97
6,dwi_hyperintensity,117,31.97
7,t1_hypointensity,114,31.15
8,hemorrhage,113,30.87
9,calcification,112,30.60


### 6b. Resolve missingness — tag structural vs MNAR

Not all `NaN` means the same thing. Before MICE imputes anything, classify each
column with missing values into one of three buckets:

| Bucket | Meaning | Action in this section |
|---|---|---|
| **Structural** | The value *does not exist* (e.g. `lesion_2_MRI_PIRADS` is NaN because the MRI showed only one lesion) | `mark_structural_missing(...)` — derives count + max features, flips originals to `kind='skip'` so MICE never touches them |
| **MNAR** | Missingness depends on the unobserved value itself (e.g. PSA not measured because the clinician judged it unnecessary) | `add_missing_flags(...)` in section 6c — adds an explicit `<col>_missing` flag, then imputes normally |
| **MAR** | Missingness depends only on *observed* variables | No special handling — MICE handles it correctly out of the box |

**Rule of thumb.** Ask: *"If this patient were re-examined today with perfect technique,
would a value exist?"* If **no** → structural. If **yes** → MAR/MNAR.

The structural step **must come before** MICE, because MICE will happily fabricate
PIRADS scores for non-existent lesions otherwise.


In [13]:
# STRUCTURAL_GROUPS — configure one entry per "slot family" in your data.
#
# A "slot family" is a set of columns that hold OPTIONAL repeats of the same
# clinical thing (lesion 1 / 2 / 3, tumour 1 / 2, biopsy core 1 / 2 / 3 …).
# NaN in slot 2 or 3 doesn't mean "we forgot to measure" — it means "that slot
# doesn't exist for this patient". Imputing it would invent findings.
#
# Each entry takes these keys:
#
#   'cols'         : list of all slot columns in the family (INCLUDE slot 1 here,
#                    so the count feature reflects the true number of slots).
#
#   'derive_count' : True  → create <group_name> = number of non-null slots.
#                    Use for "how many lesions did this patient have?"
#
#   'derive_max'   : True  → create <group_name>_max = max value across slots.
#                    Clinically the "dominant" lesion's PIRADS. Only set True
#                    when the slot columns are ORDINAL or NUMERIC.
#
#   'count_levels' : optional ordering for the count feature, e.g. [0,1,2,3].
#                    Sets the ordinal categories so charts/tables are ordered.
#
#   'max_levels'   : optional ordering for the max feature, e.g. [1,2,3,4,5].
#
#   'skip_after'   : list of columns to mark kind='skip' AFTER deriving features.
#                    IMPORTANT: usually leave the PRIMARY slot (lesion_1) OUT of
#                    this list — its value is real, not structural, and it stays
#                    as a normal predictor. Only skip the optional repeats.
#
# Effect: skipped columns are excluded from MICE imputation, EDA screening,
# the multivariable model, and the DDA second pass. They stay in the dataframe
# (you can still inspect them) but no statistic touches them.
# ─────────────────────────────────────────────────────────────────────────

STRUCTURAL_GROUPS = {
    # Not used for this meningioma run — leave empty unless you have optional
    # repeat slots (lesion 2/3, second tumour, extra biopsy core, etc.).
    #
    # Example entry (uncomment and rename columns to match your dataset):
    #
    # 'n_lesions': {
    #     'cols':         ['lesion_1_score', 'lesion_2_score', 'lesion_3_score'],
    #     'derive_count': True,   # → n_lesions = number of non-null slots
    #     'derive_max':   True,   # → n_lesions_max = max score across slots
    #     'count_levels': [0, 1, 2, 3],
    #     'max_levels':   [1, 2, 3, 4, 5],
    #     'skip_after':   ['lesion_2_score', 'lesion_3_score'],  # keep slot 1
    # },
}

if STRUCTURAL_GROUPS:
    df = mark_structural_missing(df, schema, STRUCTURAL_GROUPS)
    new_cols = [g for g in STRUCTURAL_GROUPS] + [f'{g}_max' for g in STRUCTURAL_GROUPS]
    print('Derived structural features:', [c for c in new_cols if c in df.columns])
    print('Now marked kind=\'skip\' (excluded from MICE / EDA / inferential):',
          [c for c, sp in schema.items() if sp.kind == 'skip'])
else:
    print('No structural-missing groups configured.')
    print('If your dataset has NaN that means "this slot does not exist",')
    print('edit STRUCTURAL_GROUPS above before running MICE in section 11.')


No structural-missing groups configured.
If your dataset has NaN that means "this slot does not exist",
edit STRUCTURAL_GROUPS above before running MICE in section 11.


In [14]:
# Sanity check — verify the new derived columns look right
derived = [g for g in STRUCTURAL_GROUPS] + [f'{g}_max' for g in STRUCTURAL_GROUPS]
derived = [c for c in derived if c in df.columns]
if derived:
    display(df[derived].describe(include='all'))
    for c in derived:
        print(c, df[c].value_counts(dropna=False).to_dict())

### 6c. Add MNAR missingness flags

For columns where the missingness itself carries information (true MNAR — e.g.
PSA not measured *because* risk looked low), add an explicit boolean flag so
the model can use 'was-it-measured' as a predictor. The schema is updated
automatically — no need to register the flag columns yourself.


In [15]:
# ─────────────────────────────────────────────────────────────────────────
# MNAR_COLS — list ONLY columns that meet ALL THREE criteria:
#
#   1. The value EXISTS in reality (it's not structurally absent — those went
#      into STRUCTURAL_GROUPS in section 6b).
#   2. The value is sometimes NOT recorded.
#   3. The reason it wasn't recorded is plausibly TIED TO THE VALUE ITSELF
#      (e.g. PSA not measured BECAUSE the clinician thought the patient was
#      low-risk → low-risk patients are systematically missing → MNAR).
#
# If missingness is purely due to data-entry chaos / random clerical loss,
# that's MAR — leave the column OUT of MNAR_COLS; MICE handles it correctly
# without a flag.
#
# Effect: for each column listed here, a new boolean column <col>_missing is
# added (True where the original was NaN) and registered in the schema as a
# binary predictor. The multivariable model can then use the FACT of missing-
# ness as its own predictor, separately from the imputed value.
# ─────────────────────────────────────────────────────────────────────────

MNAR_COLS = []   # e.g. ['Pirmsoperācijas_PSA', 'PSA_blīvums']

df = add_missing_flags(df, MNAR_COLS, schema=schema)
df.filter(like='_missing').head()


""
1
2
3
4
5


## 7. Derive new columns (vecums bins, time bins, PSA categories…)

Use the helpers below freely. Any new column you add **must** also be added to the
schema so DDA/EDA/inferential will analyze it.


In [16]:
# bin_numeric(s, bins, labels=None, *, right=False, ordered=True)
#   s       — numeric column to bin
#   bins    — cutpoints, e.g. [0, 50, 60, 70, 120] or [-np.inf, 90, 180, np.inf]
#   labels  — names for each bin, e.g. ['<50', '50-59', '60-69', '70+']
#   right   — False: intervals [low, high); True: (low, high]  (default False)
#   ordered — keep bin order for ordinal analysis (default True)
#
# bin_datetime(s, *, unit='year')
#   s     — datetime column (parsed with pd.to_datetime)
#   unit  — 'year' | 'quarter' | 'month' | 'week' | 'weekday' | 'hour'
#
# Derive pattern (each new column):
#   1. df['new_col'] = ...
#   2. schema['new_col'] = ColSpec(name='new_col', kind='ordinal', ordered_levels=[...])

print(df.columns)

old_df_cols = df.columns.copy()

# ── 1. Ki-67: text ranges (e.g. "1-5") → midpoint → ordinal group ─────────
if 'ki67_pct' in df.columns:
    def ki67_midpoint(x):
        if pd.isna(x):
            return pd.NA
        parts = str(x).replace(",", ".").split("-")
        nums = [float(p) for p in parts]
        return sum(nums) / len(nums)
    df["ki67_mid"] = df["ki67_pct"].map(ki67_midpoint).astype("Float64")
    def ki67_group(x):
        if pd.isna(x):
            return pd.NA
        if x <= 4:
            return "low_le_4"
        if x < 10:
            return "intermediate_5_9"
        return "high_ge_10"
    df["ki67_group"] = df["ki67_mid"].map(ki67_group)
schema['ki67_group'] = ColSpec(
        name='ki67_group', kind='ordinal',
        ordered_levels=['low_le_4', 'intermediate_5_9', 'high_ge_10'],
    )
schema['ki67_mid'] = ColSpec(
        name='ki67_mid', kind='continuous'
    )

# ── 2. age: numerical continuous -> bins ─────────
if "age" in df.columns:
    df["age_bins"] = bin_numeric(df.age, [-np.inf, 50, 60, 70, 80, +np.inf], labels=["<50", "50-59", "60-69", "70-79", "80+"], right=False, ordered=True)
schema['age_bins'] = ColSpec(
        name='age_bins', kind='ordinal',
        ordered_levels=["<50", "50-59", "60-69", "70-79", "80+"],
    )

# ── 3. adc_value: numerical continuous -> bins ─────────
#TODO: per need like in age

# ── 4. edema_volume_cm3: numerical continuous -> bins ─────────
#TODO: per need like in age

# ── 4. max_diameter_cm: numerical continuous -> bins ─────────
#TODO: per need like in age

# ── 5. tumor_volume: numerical continuous -> bins ─────────
#TODO: per need like in age

# ── 6. who_grade: ordinal categories -> bins ─────────
df['high_grade'] = df['who_grade'].isin(["2", "3"])  # True = grade 2/3

schema['high_grade'] = ColSpec(name='high_grade', kind='binary')


new_df_cols = df.columns.difference(old_df_cols)
df[new_df_cols].head()

write_cleaned_csv(OUTPUT_ROOT, df, schema)  # include derived cols in cleaned.csv

Index(['id', 'patient_code', 'entry_year', 'age', 'sex', 'histology_available',
       'who_grade', 'progesterone_pos', 'ki67_pct', 'brain_invasion',
       'hist_necrosis', 'mri_date', 'side', 'tumor_location',
       'meningioma_count', 'max_diameter_cm', 'tumor_volume', 'base_modality',
       'iv_contrast', 'tumor_episode', 'tumor_margin', 'dural_tail',
       'capsular_enhancement', 'heterogeneous_enhancement', 'perifocal_edema',
       'edema_volume_cm3', 'mass_effect', 'calcification', 'cystic_component',
       'necrosis', 'hemorrhage', 'hyperostosis', 'cortical_destruction',
       'dwi_hyperintensity', 't2_hyperintensity', 't1_hypointensity',
       'sinus_invasion', 'transfalcine_extension', 'adc_value'],
      dtype='str')


PosixPath('output/cleaning/cleaned.csv')

## 8. DDA — second pass (with derived columns)

Re-run DDA so the new columns get their own plots and stats.


In [17]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)
dda_tables['categorical']
dda_tables['continuous']

,column,kind,n,n_unique,missing_pct,min,p_5th,median,mean,trimmed_mean,p_95th,max,mode,std,cv,iqr,skewness,kurtosis
0,age,continuous,366,61,0.00,20.000,40.0000,64.50,63.021858,63.636054,81.0000,92.0,NaN,12.755641,0.202400,17.7500,-0.457413,-0.072710
1,max_diameter_cm,continuous,262,85,28.42,1.210,1.8000,3.80,4.135076,4.003810,7.4000,9.2,3.00,1.756800,0.424853,2.5000,0.602967,-0.423890
2,tumor_volume,continuous,286,251,21.86,0.189,1.7075,14.40,28.686105,22.407435,100.6500,168.0,NaN,33.117038,1.154463,32.7875,1.602898,1.947349
3,edema_volume_cm3,continuous,22,20,93.99,0.000,0.0000,17.40,30.758182,24.426667,100.7600,135.0,0.00,37.202721,1.209523,42.8750,1.375768,1.108988
4,adc_value,continuous,236,69,35.52,0.410,0.6150,0.83,0.844763,0.832084,1.1625,1.7,0.79,0.163424,0.193456,0.1700,1.072887,3.349694
5,ki67_mid,continuous,366,29,0.00,1.000,1.0000,2.50,4.299180,2.965986,17.5000,55.0,1.00,5.785312,1.345678,3.5000,3.970971,22.016304


In [18]:
df.columns

Index(['id', 'patient_code', 'entry_year', 'age', 'sex', 'histology_available',
       'who_grade', 'progesterone_pos', 'ki67_pct', 'brain_invasion',
       'hist_necrosis', 'mri_date', 'side', 'tumor_location',
       'meningioma_count', 'max_diameter_cm', 'tumor_volume', 'base_modality',
       'iv_contrast', 'tumor_episode', 'tumor_margin', 'dural_tail',
       'capsular_enhancement', 'heterogeneous_enhancement', 'perifocal_edema',
       'edema_volume_cm3', 'mass_effect', 'calcification', 'cystic_component',
       'necrosis', 'hemorrhage', 'hyperostosis', 'cortical_destruction',
       'dwi_hyperintensity', 't2_hyperintensity', 't1_hypointensity',
       'sinus_invasion', 'transfalcine_extension', 'adc_value', 'ki67_mid',
       'ki67_group', 'age_bins', 'high_grade'],
      dtype='str')

## 9. Configure targets and predictors

This is the only place outcome variables and candidate predictors are declared.


In [19]:
EDA_TARGETS = ['high_grade',]
EDA_PREDICTORS = ['age', 'sex', 'histology_available', 'progesterone_pos', 'brain_invasion',
       'hist_necrosis', 'side', 'tumor_location',
       'meningioma_count', 'max_diameter_cm', 'tumor_volume', 'base_modality',
       'iv_contrast', 'tumor_episode', 'tumor_margin', 'dural_tail',
       'capsular_enhancement', 'heterogeneous_enhancement', 'perifocal_edema',
       'edema_volume_cm3', 'mass_effect', 'calcification', 'cystic_component',
       'necrosis', 'hemorrhage', 'hyperostosis', 'cortical_destruction',
       'dwi_hyperintensity', 't2_hyperintensity', 't1_hypointensity',
       'sinus_invasion', 'transfalcine_extension', 'adc_value', 'ki67_mid',
       'ki67_group', 'age_bins']

INFERENTIAL_TARGETS = ['high_grade']
INFERENTIAL_PREDICTORS = [
    #'sex',
    #'histology_available',
    #'progesterone_pos',
    #'brain_invasion',
    #'hist_necrosis',
    #'side', 
    #'tumor_location',
    #'meningioma_count',
    'max_diameter_cm',
    #'tumor_volume',
    #'base_modality',
    #'iv_contrast',
    #'tumor_episode',
    #'tumor_margin',
    #'dural_tail',
    'capsular_enhancement',
    #'heterogeneous_enhancement',
    #'perifocal_edema',
    #'edema_volume_cm3',
    #'mass_effect',
    'cystic_component',
    #'necrosis',
    #'hemorrhage',
    #'hyperostosis',
    'cortical_destruction',
    #'dwi_hyperintensity',
    #'t2_hyperintensity',
    't1_hypointensity',
    #'sinus_invasion',
    #'transfalcine_extension',
    'adc_value', 
    
    #'ki67_pct',   --> not gonna be analysed anyway
    #'ki67_mid',
    #'ki67_group',
    
    #'age_bins',
    #'age', 
    ]

#TARGETS = ['no_biopsijas_līdz_RPE_mēneši']
#PREDICTORS = ['no_biopsijas_līdz_RPE_mēneši']

EDA_PREDICTORS = [c for c in EDA_PREDICTORS if c in df.columns]  # drop any missing names
INFERENTIAL_PREDICTORS = [c for c in INFERENTIAL_PREDICTORS if c in df.columns]  # drop any missing names

# Binary targets only: which value counts as the event (positive class).
EDA_POSITIVE_CLASS = {t: True for t in EDA_TARGETS if t in (
    'histology_available', 'progesterone_pos', 'brain_invasion',
    'hist_necrosis', 'iv_contrast', 'tumor_margin', 'dural_tail',
    'capsular_enhancement', 'heterogeneous_enhancement', 'perifocal_edema',
    'mass_effect', 'calcification', 'cystic_component',
    'necrosis', 'hemorrhage', 'hyperostosis', 'cortical_destruction',
    'dwi_hyperintensity', 't2_hyperintensity', 't1_hypointensity',
    'transfalcine_extension',)}
INFERENTIAL_POSITIVE_CLASS = {t: True for t in INFERENTIAL_TARGETS if t in (
    'histology_available', 'progesterone_pos', 'brain_invasion',
    'hist_necrosis', 'iv_contrast', 'tumor_margin', 'dural_tail',
    'capsular_enhancement', 'heterogeneous_enhancement', 'perifocal_edema',
    'mass_effect', 'calcification', 'cystic_component',
    'necrosis', 'hemorrhage', 'hyperostosis', 'cortical_destruction',
    'dwi_hyperintensity', 't2_hyperintensity', 't1_hypointensity',
    'transfalcine_extension',)}

## 10. EDA — univariate screening

Targets can be **binary**, **continuous**, **ordinal**, or **nominal** (from schema). The test depends on both outcome and predictor types — e.g. ordinal outcome × nominal predictor → χ²; continuous outcome × nominal predictor → Kruskal–Wallis.

| target kind   | continuous / count predictor | ordinal predictor | nominal / binary predictor |
|---------------|------------------------------|-------------------|----------------------------|
| binary        | Mann–Whitney U               | Spearman ρ        | χ² / Fisher                |
| continuous    | Spearman ρ                   | Spearman ρ        | Kruskal–Wallis             |
| ordinal       | Spearman ρ                   | Spearman ρ        | χ²                         |
| nominal       | Kruskal–Wallis               | χ²                | χ²                         |

`POSITIVE_CLASS` applies only to **binary** targets. Multivariable logistic (§11) remains **binary outcomes only**.

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [20]:
assoc = screen_associations(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

assoc[assoc['fdr_significant']]


,target,target_kind,predictor,kind,test,stat,p,p_fdr,fdr_significant,effect,effect_label,n_used,positive_class
0,high_grade,binary,ki67_group,ordinal,spearman,0.693733,7.917526e-54,2.850309e-52,True,0.693733,spearman_rho,366,True
1,high_grade,binary,ki67_mid,continuous,mann_whitney_u,26262.500000,4.744216e-40,8.539589e-39,True,-0.865234,rank_biserial_r,366,True
2,high_grade,binary,hist_necrosis,binary,chi2,63.039325,2.026206e-15,2.431447e-14,True,0.415016,cramers_v,366,True
3,high_grade,binary,tumor_volume,continuous,mann_whitney_u,12186.000000,2.201526e-07,1.981374e-06,True,-0.381633,rank_biserial_r,286,True
4,high_grade,binary,max_diameter_cm,continuous,mann_whitney_u,9908.500000,2.143737e-05,1.543491e-04,True,-0.325375,rank_biserial_r,262,True
5,high_grade,binary,hyperostosis,binary,chi2,15.247352,9.430889e-05,5.658533e-04,True,0.243574,cramers_v,257,True
6,high_grade,binary,tumor_location,nominal,chi2,13.105145,2.944859e-04,1.514499e-03,True,0.223651,cramers_v,262,True
7,high_grade,binary,cystic_component,binary,chi2,12.285057,4.565994e-04,1.994370e-03,True,0.218636,cramers_v,257,True
8,high_grade,binary,tumor_margin,nominal,chi2,12.120922,4.985924e-04,1.994370e-03,True,0.214679,cramers_v,263,True
9,high_grade,binary,perifocal_edema,binary,chi2,11.783955,5.974341e-04,2.150763e-03,True,0.212484,cramers_v,261,True


In [21]:
# Full table
#assoc


## 11. Multiple imputation (MICE)

We generate **m=10** imputed datasets via sklearn's IterativeImputer
(RandomForest estimator, separate random seed per imputation).
The pooled inferential stage applies Rubin's rules over these 10 fits.

For a quick screening run set `m=3`. For publication use `m≥10`.


In [22]:
M = 3  # number of imputations; reduce to 3 for fast iteration

#imputed_frames = mice_impute(df, schema, m=M, max_iter=10,
#                             random_state=42, output_root=OUTPUT_ROOT)
#print(f"Generated {len(imputed_frames)} imputed frames")
#print("NaN count in first imputed frame:", imputed_frames[0].isna().sum().sum())


## 12. Multivariable logistic regression (Rubin-pooled)

For each target:

1. Build design matrix (continuous z-scored, ordinal kept as codes, nominal one-hot).
2. Iteratively drop predictors with **VIF > 5** to handle collinearity.
3. Fit logistic regression on each of the m imputed frames.
4. Pool coefficients with **Rubin's rules** (Barnard–Rubin df).
5. Report adjusted OR with 95% CI and pooled p-value.
6. Save a forest plot SVG per target.


In [23]:
# Skip MICE for now — quick median/mode imputation (screening only)
imputed_frames = [simple_impute(df, schema)]
print("NaN count:", imputed_frames[0].isna().sum().sum())

inf_results = run_inferential(
    imputed_frames, schema,
    targets=INFERENTIAL_TARGETS,
    predictors=INFERENTIAL_PREDICTORS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    vif_threshold=5.0,
    output_root=OUTPUT_ROOT,
)
#inf_results

NaN count: 64


In [24]:
# Significant adjusted predictors per target (binary outcomes only)
if inf_results.empty:
    print("No multivariable results — §11 logistic needs a binary TARGETS entry "
          "(e.g. upgrade). Ordinal/continuous targets are EDA-only (§10).")
else:
    inf_results[(inf_results["p"] < 0.05) & inf_results["or"].notna()]


## 13. **REPORT**


In [25]:
from report import build_report, ReportConfig, write_html
from pathlib import Path

# Primary MRI/histology predictor — dedicated stats + figures in Final conclusion
FOCUS_PREDICTOR = "age_bins"
# Reference category for nominal/binary focus vars (ignored for continuous).
FOCUS_REFERENCE_LEVEL = None  # e.g. "irregular" for tumor_margin

_report_title = (
    "Preoperative MRI and Histology in Relation to "
    "WHO 2021 Meningioma Grade"
)
if ANALYSIS_YEARS is not None:
    _report_title += f" ({ANALYSIS_YEARS} cohort)"

# Two-panel focus figure (overall counts + within-year shares) when cohort spans years.
if ANALYSIS_YEARS is None and FOCUS_PREDICTOR and YEAR_COLUMN in df.columns:
    _by_year = plot_distribution_by_year(
        df, FOCUS_PREDICTOR, YEAR_COLUMN, OUTPUT_ROOT / "dda" / "figures"
    )
    if _by_year:
        print(f"Focus by-year figure: {_by_year}")

cfg = ReportConfig(
    output_root=Path("output"),
    title=_report_title,
    author="radio team",
    targets=tuple(EDA_TARGETS),
    focus_predictor=FOCUS_PREDICTOR,
    focus_reference_level=FOCUS_REFERENCE_LEVEL,
    year_column=YEAR_COLUMN,
)
report_path = Path("output/report/report.html")
write_html(build_report(cfg), report_path)
print(f"Report written: {report_path.resolve()}")

Focus by-year figure: output/dda/figures/age_bins__bar_by_year.svg
Report written: /Users/andriszaguzovs/TheLibraryOfCode/meningioma-atypier/heavy_machinery/output/report/report.html


## 13. Outputs

Everything is saved on disk:

```
output/
├── dda/{figures,tables}/
├── missingness/{figures,tables}/
├── eda/{figures,tables}/
└── inferential/{figures,tables}/
```

Each plot is an individual `.svg`; each result table an individual `.csv`.


In [26]:
from pathlib import Path
for p in sorted(Path(OUTPUT_ROOT).rglob('*')):
    if p.is_file():
        print(p)


output/cleaning/cleaned.csv
output/cleaning/cleaning_log.csv
output/cleaning/cleaning_summary.csv
output/dda/figures/Perifokālas tūskas tilpums, cm3__box.svg
output/dda/figures/Perifokālas tūskas tilpums, cm3__hist.svg
output/dda/figures/adc_value__box.svg
output/dda/figures/adc_value__hist.svg
output/dda/figures/age__box.svg
output/dda/figures/age__hist.svg
output/dda/figures/age_bins__bar.svg
output/dda/figures/age_bins__bar_by_year.svg
output/dda/figures/base_modality__bar.svg
output/dda/figures/brain_invasion__bar.svg
output/dda/figures/calcification__bar.svg
output/dda/figures/capsular_enhancement__bar.svg
output/dda/figures/cortical_destruction__bar.svg
output/dda/figures/cystic_component__bar.svg
output/dda/figures/dural_tail__bar.svg
output/dda/figures/dwi_hyperintensity__bar.svg
output/dda/figures/edema_volume_cm3__box.svg
output/dda/figures/edema_volume_cm3__hist.svg
output/dda/figures/hemorrhage__bar.svg
output/dda/figures/heterogeneous_enhancement__bar.svg
output/dda/figure

In [27]:
# number of high-grade events
display(df["high_grade"].value_counts(dropna=False))

# rows actually used in model
print(df.shape)

# predictors per event rough check
n_events = df["high_grade"].sum()
n_predictors = len(INFERENTIAL_PREDICTORS)
print(n_events / n_predictors)              # ==> <5 means the multivariate model is unstable - decrease the number of variables

high_grade
False    256
True     110
Name: count, dtype: int64

(366, 43)
18.333333333333332


## 14. NOTES — Why each statistical choice

Concise but detailed rationale for every formula used in this pipeline.
For each: **what it does**, **why chosen**, **what was rejected**.

---

### Schema inference (hybrid auto + override)

- **What.** Heuristic classification of each column into `continuous / count / ordinal / nominal / binary / datetime / id / text / skip` based on dtype, cardinality, value patterns.
- **Why.** Test selection downstream is kind-driven — a wrong kind silently picks the wrong test (e.g. treating Gleason 1–5 as `continuous` instead of `ordinal` swaps Spearman for MWU and loses interpretability of "per-grade increase").
- **Alternatives rejected.**
  - *Full auto-only*: brittle on clinical data where 0/1-coded ordinals look numeric.
  - *Manual ColSpec per column*: correct but tedious; you'd re-type 30+ specs per study.

---

### Duplicate auditing on normalized string keys

- **What.** Lowercase + strip + empty→NA on ID columns, then flag rows whose full key tuple is non-null and repeated.
- **Why.** Clinical IDs (`pk`, `year`) frequently have invisible whitespace or case drift across data-entry sessions. Naive `duplicated()` misses these.
- **Alternatives rejected.**
  - *Exact match*: under-detects.
  - *Fuzzy match (Levenshtein)*: over-detects, would falsely merge genuinely different patients.

---

### Mann–Whitney U for continuous/count vs binary outcome

- **What.** Non-parametric rank-sum test. H₀: P(X₁ > X₂) = ½. Two-sided.
- **Effect size.** Rank-biserial **r = |Z|/√N**, where Z is the large-sample normal approximation of U. Bounded 0–1, interpretable like Cohen's r (0.1 small, 0.3 medium, 0.5 large).
- **Why.**
  - Clinical continuous variables (PSA, vecums, days-to-surgery) are **almost never normal** — PSA in particular is heavily right-skewed.
  - MWU has ~95% efficiency vs t-test under normality and is far more robust under non-normality.
  - One test for the whole pipeline = no test-switching artifacts.
- **Alternatives rejected.**
  - *Welch's t-test always*: violates assumption on skewed data; inflates type-I error on small skewed samples.
  - *Auto Shapiro-Wilk switch (t if normal, MWU else)*: the normality test itself adds noise and its decision is sample-size dependent (always rejects normal at large N, never at small N) — produces worse calibration than just using MWU.
  - *Welch's t on log-transformed data*: works for PSA specifically but not generalizable to all continuous predictors in the pipeline.
- **Sensitivity.** When publishing, re-run Welch's t on log(PSA) as a sensitivity analysis — if direction and significance agree with MWU, you're robust.

---

### Spearman ρ for ordinal vs binary outcome

- **What.** Pearson correlation on the ranks of category codes vs the 0/1-encoded outcome.
- **Why.**
  - Preserves the **ordering** of ordinal predictors (Gleason 1<2<3<4<5, PIRADS 1<2<3<4<5, risk_group low<mid<high). χ² throws this away — it would only tell you "the distribution differs across levels", not "higher Gleason → more upgrades".
  - Yields a signed, scale-free effect size (ρ) that's directly publishable.
- **Alternatives rejected.**
  - *χ² on the ordinal × binary table*: ignores ordering, weaker power, no direction.
  - *Cochran-Armitage trend test*: equivalent to a linear-trend variant of χ² and gives p only — Spearman gives p **plus** a comparable ρ across all ordinal predictors.
  - *Kendall's τ*: similar info but slower on large N and no power advantage here.

---

### χ² (or Fisher exact) for nominal vs binary

- **What.** χ² of independence on the contingency table, **without Yates correction** (modern recommendation — Yates is overconservative). Switches to **Fisher exact** if the 2×2 table has any expected cell count < 5.
- **Why Fisher when expected<5.** χ²'s asymptotic distribution breaks down with small expected counts; Fisher's exact test conditions on the marginals and computes the exact hypergeometric p — correct at any sample size.
- **Effect size: Cramér's V** = √(χ²/(N·(min(r,c)−1))). Bounded 0–1, comparable across table shapes. For 2×2 tables we **also** report the odds ratio because clinicians read OR natively.
- **Alternatives rejected.**
  - *Yates-corrected χ²*: too conservative for modern computing — Fisher is exact and almost as fast.
  - *G-test (likelihood ratio)*: theoretically nicer for nested models but identical conclusions in 2-way tables; less familiar to clinical readers.
  - *Permutation χ²*: same answer as Fisher for 2×2, more expensive.

---

### Benjamini–Hochberg FDR correction, per target

- **What.** Sort p-values ascending; for rank i out of m, compute q_i = p_(i)·m/i; enforce monotonicity from the right; significance at q < α controls expected proportion of false discoveries at α.
- **Why per-target (not pooled across all targets).** Each outcome (upgrade, upstage, downgrade) is a **separate family** of hypotheses with its own scientific interpretation. Pooling them inflates the family size and over-corrects. This matches how clinical journals report multi-outcome studies.
- **Alternatives rejected.**
  - *Bonferroni*: controls family-wise error rate — far too conservative for a screening stage with 10+ predictors. Misses real signal.
  - *Holm-Bonferroni*: still FWER, marginally less conservative than Bonferroni but still much stricter than BH.
  - *Storey q-value*: estimates the null proportion adaptively; great when you have hundreds of tests but unstable at small m (you'll have <20 tests per target).
  - *No correction*: indefensible with ≥3 predictors per target — false discovery rate would be ~30%+.
- **Verified.** Output matches `statsmodels.stats.multitest.multipletests(method='fdr_bh')` exactly.

---

### MICE (Multiple Imputation by Chained Equations), m=10

- **What.** For each missing value: fit a regression of that column on all others using observed data, predict missing values, iterate until convergence. Repeat with m different random seeds to produce m plausible completed datasets.
- **Why multiple (not single).** Single imputation pretends the imputed values are known, so it **understates standard errors**. With m=10 imputations and Rubin pooling, the SEs honestly include imputation uncertainty.
- **Estimator: RandomForestRegressor.** Captures non-linear relationships (PSA × vecums × Gleason interactions) without you specifying them. Tolerates mixed numeric/categorical inputs.
- **Why m=10.** Rubin showed efficiency = (1 + fmi/m)^(-1) where fmi is fraction of missing info. At fmi ≈ 0.3 (typical clinical data), m=10 gives ~97% efficiency. m=5 is acceptable, m=20 is overkill.
- **Alternatives rejected.**
  - *Mean/median imputation*: distorts variance and any correlation involving the imputed column. Catastrophic for inferential SEs.
  - *Complete-case analysis*: throws away rows with any missingness — typically 20–50% data loss in clinical cohorts; introduces selection bias if missingness is MAR (which it usually is).
  - *Hot-deck imputation*: works for nominal-only data; weaker for mixed types.
  - *Bayesian model-based imputation (`mice` R package, Stan)*: gold standard but heavy infrastructure; sklearn's `IterativeImputer` is close enough for clinical screening.
- **Limitation.** Assumes data is **Missing At Random** (MAR) — missingness depends only on observed variables. For **MNAR** patterns (e.g. "PSA was missing because risk was low"), add explicit `<col>_missing` flags in section 6a so the model can use the missingness indicator itself as a predictor.

---

### Missingness flags

- **What.** Binary indicator columns `<col>_missing` added before imputation.
- **Why.** In clinical data, *that a value was missing* is often informative (e.g. PSA not measured because clinician judged it unnecessary). Including the flag in the regression lets the model separate "the value's effect" from "the act of measuring's effect".
- **Alternatives rejected.**
  - *Imputing without flags*: hides the MNAR mechanism.
  - *Dropping columns with high missingness*: throws away signal; missingness % is not a reliable filter for clinical utility.

---

### Variance Inflation Factor (VIF) pruning, threshold = 5

- **What.** For each predictor x_j, VIF = 1/(1 − R²_j), where R²_j is from regressing x_j on all other predictors. Iteratively drop the column with the highest VIF until all ≤ 5.
- **Why.** Logistic regression with collinear predictors produces enormous standard errors and unstable coefficients ("model can't tell whether PSA or PSA-density is doing the work"). VIF > 5 ⇔ R²_j > 0.80 ⇔ severe multicollinearity.
- **Why threshold = 5** (not 10). VIF=10 is the classical statistics teaching threshold but for clinical regression with modest N (<500), 5 is the modern recommendation (Vatcheva 2016, O'Brien 2007).
- **Alternatives rejected.**
  - *Pairwise Pearson correlation > 0.8*: catches only 2-variable collinearity; misses 3-way (e.g. a = b + c).
  - *Lasso regularization*: would drop collinear features automatically but **biases coefficients** toward zero — bad for inference (you want unbiased OR estimates). Lasso is for prediction, not inference.
  - *Ridge / Elastic Net*: same problem — shrinks coefficients, distorts ORs.
  - *PCA / partial-least-squares*: components are uninterpretable clinically.

---

### Multivariable logistic regression

- **What.** Per target, one binary logistic model with all surviving predictors. Continuous z-scored (so OR is per-SD increase), ordinals kept as numeric codes, nominals one-hot with drop_first.
- **Why.** Univariate screening (section 10) ignores confounding — Gleason can show up "significant" purely because it correlates with PSA. Multivariable estimates the **adjusted** effect of each predictor holding the others constant.
- **Why this design encoding.**
  - *z-score continuous*: ORs comparable across predictors; one "unit" = one SD.
  - *Ordinal as numeric code*: assumes linear log-odds across levels (parsimonious; standard for Gleason/PIRADS in urology papers). The alternative is one-hot with drop_first, which uses more degrees of freedom and is only worth it if the trend is clearly non-monotonic — check the EDA plots first.
  - *Nominal one-hot drop_first*: avoids the dummy variable trap (perfect collinearity with intercept).
- **Alternatives rejected.**
  - *Univariate-only pipeline*: misleading because of confounding.
  - *Stepwise selection (forward/backward)*: notorious for unstable selection, inflated significance, and irreproducibility. Modern guidance (Harrell, Steyerberg) is: don't.
  - *Random forest / XGBoost*: better predictive accuracy but no clinical OR with CI to report.
  - *Penalized regression (Firth, Lasso, Ridge)*: useful with extreme separation or n<<p but biases the OR estimates — defeats the inferential purpose.
  - *Bayesian logistic with weakly informative priors*: cleaner for tiny samples and would give credible intervals — but you'd need to defend prior choice in the manuscript.

---

### Rubin's rules with Barnard–Rubin degrees of freedom

- **What.** Across the m=10 imputed-frame fits, for each coefficient:
  - θ̄ = mean of the m point estimates
  - within-imp variance Ū = mean of the m squared SEs
  - between-imp variance B = sample variance of the m estimates
  - total variance T = Ū + (1 + 1/m)·B
  - pooled SE = √T
  - degrees of freedom (Barnard–Rubin):
    df = (m−1)·(1 + Ū/((1+1/m)·B))²
  - p-value from t-distribution with that df; 95% CI = θ̄ ± t_{0.975, df}·SE
- **Why.** Rubin's rules are the **only** statistically valid way to combine results across multiple imputations. The total variance T splits into "within" (each model's uncertainty) and "between" (uncertainty due to missing data) — they're not interchangeable.
- **Why Barnard–Rubin df (not the original Rubin 1987 df).** Original Rubin df → ∞ when between-variance is small, which is wrong when m is small. Barnard–Rubin (1999) is a small-sample correction that's now the standard (R `mice` uses it, SAS PROC MIANALYZE uses it).
- **Alternatives rejected.**
  - *Picking the "best" imputation*: defeats the purpose of multiple imputation entirely.
  - *Average the imputed datasets first, then fit once*: produces correct point estimates but **wrong SEs** (the between-variance is invisible).
  - *Use the within-variance only*: ignores imputation uncertainty — false confidence.
- **Verified.** With zero between-variance, our pooler returns SE equal to the single-fit SE; with non-zero between, it correctly inflates SE and produces a finite small-sample df (e.g. m=5, modest B → df ≈ 22).

---

### Why log-scale x-axis on forest plots

- ORs are multiplicative (OR=2 and OR=0.5 are equal-and-opposite effects). On a linear axis they look asymmetric; on log scale they're symmetric around OR=1, which is the correct visual.

---

### What this pipeline deliberately does NOT do

- **No machine-learning prediction** (no train/test split, no AUC, no calibration). This is an **association/inference** pipeline, not a prediction pipeline. If you later want a predictive model (e.g. nomogram for upgrade risk), that's a separate workflow with cross-validation, calibration plots, decision-curve analysis.
- **No causal inference** (no DAGs, no IPTW, no instrumental variables). All effects here are **statistical associations** adjusted for the included covariates — they are *not* causal effects. Manuscript wording must say "associated with", never "causes".
- **No survival/time-to-event analysis.** Targets here are binary (upgrade yes/no). If you later care about *time to biochemical recurrence*, you'd need Cox regression — a separate module.

---

### Sanity-check checklist before submitting results

1. Print `schema_summary(schema)` — every ordinal has correct `ordered_levels`?
2. After MICE, `imputed_frames[0].isna().sum().sum()` == 0 for predictor columns?
3. `inf_results['n_models']` ≈ m for all predictors (means the model converged on every imputation)?
4. Forest plot ORs and EDA univariate effects agree in **direction** (sign)? If they flip, you have confounding worth discussing.
5. For each FDR-significant univariate result, check the corresponding plot in `output/eda/figures/` — is the pattern visually credible or driven by 2–3 outliers?
